# Data Preprocessing & Cleaning
## National Transport Decarbonisation Dashboard for Ireland

This notebook implements the **Integration** stage of the Data Value Map from the
project proposal: it turns the raw CSO / data.gov.ie extracts into a **governed,
analysis-ready** dataset for the dashboard.

For each source it follows the same disciplined pattern — **profile → clean →
validate** — then integrates everything into an annual fact table carrying the
three composite KPIs:

| KPI | Definition | Sources |
|---|---|---|
| **Car Dependency Index** | private cars per 1,000 population | THA18 ÷ PEA01 |
| **Public Transport Usage Index** | (bus + rail + Luas) journeys per capita | THA25 + TOA11 ÷ PEA01 |
| **Transport Intensity Indicator** | total vehicle-km per capita | THA17 ÷ PEA01 |

**Core analysis window: 2019–2023** (common overlap across all core datasets).

> ⚠️ Three cleaning hazards in this data are handled explicitly below: TEM23
> mixes detail rows with `All …` subtotals (double-counting risk), PEA01 mixes
> overlapping age bands, and THA25 is missing all 2019 rail data.

## 1. Setup, paths and configuration

Paths resolve relative to the repository root. Point the pipeline at a fresh CSO
download by setting the `TDD_RAW_DIR` environment variable; otherwise it reads
from `data/raw/`. Raw files are matched by **prefix** so timestamped filenames
work without edits.

In [17]:
import os, re, glob
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

# --- paths ---
CWD = Path.cwd()
REPO_ROOT = CWD.parent if CWD.name == "notebooks" else CWD
RAW_DIR = Path(os.environ.get("TDD_RAW_DIR", REPO_ROOT / "data" / "raw"))
PROCESSED_DIR = REPO_ROOT / "data" / "processed"
DOCS_DIR = REPO_ROOT / "docs"
for d in (PROCESSED_DIR, DOCS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("Repo root :", REPO_ROOT)
print("Raw dir   :", RAW_DIR)
print("Processed :", PROCESSED_DIR)

Repo root : c:\Users\srb10\Desktop\MSc BA Project groups [Semester - 2]\IS6611\TDD Dashboard\Transport_Decarbonisation_Dashboard_IE
Raw dir   : c:\Users\srb10\Desktop\MSc BA Project groups [Semester - 2]\IS6611\TDD Dashboard\Transport_Decarbonisation_Dashboard_IE\data\raw
Processed : c:\Users\srb10\Desktop\MSc BA Project groups [Semester - 2]\IS6611\TDD Dashboard\Transport_Decarbonisation_Dashboard_IE\data\processed


In [18]:
# --- canonical configuration (from proposal section 4) ---
CORE_WINDOW = (2019, 2023)

RAW_PREFIXES = {
    "TOA11": "TOA11", "PEA01": "PEA01", "THA25": "THA25", "THA18": "THA18",
    "THA17": "THA17", "TEM12": "TEM12", "TEM22": "TEM22", "TEM23": "TEM23",
    "NAPTAN": "NaPTAN_Stop_Points",
}

# Clean NON-OVERLAPPING five-year partition of PEA01 age groups (the raw table
# also ships overlapping broad/convenience bands which must NOT be summed).
PEA01_FIVE_YEAR_BANDS = [
    "Under 1 year", "1 - 4 years", "5 - 9 years", "10 - 14 years",
    "15 - 19 years", "20 - 24 years", "25 - 29 years", "30 - 34 years",
    "35 - 39 years", "40 - 44 years", "45 - 49 years", "50 - 54 years",
    "55 - 59 years", "60 - 64 years", "65 - 69 years", "70 - 74 years",
    "75 - 79 years", "80 - 84 years", "85 years and over",
]

FUEL_GROUP_MAP = {
    "Petrol": "Petrol", "Diesel": "Diesel",
    "Electric": "Battery Electric (BEV)",
    "Petrol and electric hybrid": "Hybrid (HEV)",
    "Diesel and electric hybrid": "Hybrid (HEV)",
    "Petrol or Diesel plug-in hybrid electric": "Plug-in Hybrid (PHEV)",
    "Other fuel types": "Other",
}
ELECTRIFIED_GROUPS = {"Battery Electric (BEV)", "Hybrid (HEV)", "Plug-in Hybrid (PHEV)"}
PLUGIN_GROUPS = {"Battery Electric (BEV)", "Plug-in Hybrid (PHEV)"}
MONTH_NAME_TO_NUM = {m: i for i, m in enumerate(
    ["january","february","march","april","may","june","july","august",
     "september","october","november","december"], start=1)}

# A simple audit log rendered into the governance report at the end.
AUDIT = []
def audit(dataset, msg): AUDIT.append(f"- **{dataset}** — {msg}")

## 2. Shared helper functions

These handle the realities of CSO PxStat exports: a UTF-8 BOM on some files,
fully-quoted variants, suppressed/blank value markers, and **four** different
period formats (`2019M01`, `2022 January`, `2015 April`, `2019 Week 01` — with a
lowercase `week` variant lurking in THA25).

In [19]:
SUPPRESSED = {"", "-", "..", ":", "n/a", "na", "c", "*", "x"}

def load_raw(key):
    """Load a raw CSO CSV by prefix; strip BOM, quotes and stray whitespace."""
    matches = sorted(RAW_DIR.glob(f"{RAW_PREFIXES[key]}*.csv"))
    if not matches:
        raise FileNotFoundError(f"No file matching {RAW_PREFIXES[key]}*.csv in {RAW_DIR}")
    df = pd.read_csv(matches[-1], dtype=str, keep_default_na=False)
    df.columns = [c.replace("\ufeff", "").strip().strip('"') for c in df.columns]
    for c in df.columns:
        df[c] = df[c].str.strip()
    return df

def coerce_value(s):
    """Raw VALUE string -> float; suppressed/blank -> NaN; strip thousands sep."""
    s = s.astype(str).str.strip().str.replace(",", "", regex=False)
    s = s.where(~s.str.lower().isin(SUPPRESSED), other=np.nan)
    return pd.to_numeric(s, errors="coerce")

def blank_mask(s):
    return s.astype(str).str.strip().str.lower().isin(SUPPRESSED)

def parse_month(period):
    """'2019M01'->(2019,1); '2022 January'/'2015 April'->(y,m)."""
    p = str(period).strip()
    m = re.fullmatch(r"(\d{4})M(\d{1,2})", p, flags=re.IGNORECASE)
    if m: return int(m.group(1)), int(m.group(2))
    m = re.fullmatch(r"(\d{4})\s+([A-Za-z]+)", p)
    if m: return int(m.group(1)), MONTH_NAME_TO_NUM[m.group(2).lower()]
    raise ValueError(f"Unrecognised month period: {period!r}")

def parse_week(period):
    """'2019 Week 01' (case-insensitive) -> (2019, 1)."""
    m = re.fullmatch(r"(\d{4})\s+Week\s+(\d{1,2})", str(period).strip(),
                     flags=re.IGNORECASE)
    if not m: raise ValueError(f"Unrecognised week period: {period!r}")
    return int(m.group(1)), int(m.group(2))

def add_month_columns(df, period_col):
    ym = df[period_col].map(parse_month)
    df = df.copy()
    df["Year"]     = [y for y, _ in ym]
    df["MonthNum"] = [m for _, m in ym]
    df["Date"]     = pd.to_datetime(dict(year=df["Year"], month=df["MonthNum"], day=1))
    return df

print("Helpers ready. Raw files found:")
for k in RAW_PREFIXES:
    hits = sorted(RAW_DIR.glob(f"{RAW_PREFIXES[k]}*.csv"))
    print(f"  {k:7} -> {hits[0].name if hits else 'MISSING'}")

Helpers ready. Raw files found:
  TOA11   -> TOA11.20260528T000523 2018-2025.csv
  PEA01   -> PEA01.20260528T000539 2018-2025.csv
  THA25   -> THA25.20260528T000529 2019-2025.csv
  THA18   -> THA18.20260528T000504 2018-2023.csv
  THA17   -> THA17.20260528T000501 2018-2023.csv
  TEM12   -> TEM12.20260527T230534 2015-2026.csv
  TEM22   -> TEM22.20260528T010512 2019-2021.csv
  TEM23   -> TEM23.20260528T000533 2022-2026.csv
  NAPTAN  -> NaPTAN_Stop_Points.csv


## 3. PEA01 — Annual population

The denominator for every per-capita KPI. PEA01 reports in **thousands** and
ships *overlapping* age bands (`0–14`, `0–4`, `15 and over`, plus five-year
bands). Summing the raw table would multiply-count, so we sum a clean five-year
partition across both sexes and validate it against the broad bands.

In [20]:
pop_raw = load_raw("PEA01")
pop_raw["VALUE"] = coerce_value(pop_raw["VALUE"])
pop_raw["Year"]  = pop_raw["Year"].astype(int)

# Validation: five-year partition vs broad-band partition should agree.
five = pop_raw[pop_raw["Age Group"].isin(PEA01_FIVE_YEAR_BANDS)].groupby("Year")["VALUE"].sum()
broad_bands = ["0 - 14 years","15 - 24 years","25 - 44 years","45 - 64 years","65 years and over"]
broad = pop_raw[pop_raw["Age Group"].isin(broad_bands)].groupby("Year")["VALUE"].sum()
print("Max abs diff (five-year vs broad partition), thousands:",
      round(float((five - broad).abs().max()), 3))

population = (five.reset_index().rename(columns={"VALUE": "population_thousands"}))
population["population"] = (population["population_thousands"] * 1000).round().astype("int64")
population = population[["Year", "population"]].sort_values("Year").reset_index(drop=True)
audit("PEA01", f"population from {len(PEA01_FIVE_YEAR_BANDS)} five-year bands x 2 sexes; "
               f"years {population.Year.min()}-{population.Year.max()}.")
print(population.to_string(index=False))

Max abs diff (five-year vs broad partition), thousands: 0.4
 Year  population
 2018     4885000
 2019     4958300
 2020     5029900
 2021     5074800
 2022     5183600
 2023     5281300
 2024     5380500
 2025     5458500


## 4. TOA11 — Luas passenger numbers

Monthly journeys by line (Red / Green). Annual Luas total = both lines summed
over all 12 months. We flag any year without a full 12 months.

In [21]:
luas_raw = load_raw("TOA11")
luas_raw["VALUE"] = coerce_value(luas_raw["VALUE"])
luas_raw["Year"]  = luas_raw["Year"].astype(int)
print("Lines:", sorted(luas_raw["Statistic Label"].unique()))

months_per_year = luas_raw.groupby("Year")["Month"].nunique()
luas = (luas_raw.groupby("Year", as_index=False)["VALUE"].sum()
        .rename(columns={"VALUE": "luas_journeys"}))
luas["luas_complete"] = luas["Year"].map(lambda y: int(months_per_year.get(y, 0)) == 12)
luas["luas_journeys"] = luas["luas_journeys"].astype("int64")
for yr, n in months_per_year[months_per_year < 12].items():
    audit("TOA11", f"year {yr} has only {n}/12 months — flagged incomplete.")
print(luas.to_string(index=False))

Lines: ['Green line', 'Red line']
 Year  luas_journeys  luas_complete
 2018       41836966           True
 2019       48347231           True
 2020       19176057           True
 2021       19481318           True
 2022       38667875           True
 2023       48205217           True
 2024       54227600           True
 2025       55131012           True


## 5. THA25 — Public transport journeys (bus + rail, weekly)

Weekly passenger journeys for bus and rail (Luas is separate, in TOA11). Three
issues to handle:

1. **`Week 53` placeholders** in 52-week years are empty → dropped.
2. **2019 has no rail at all** (the weekly tracker added rail later) → 2019 is
   bus-only and flagged `rail_complete = False`.
3. Weeks summed with `min_count=1` so an all-blank mode/year stays `NaN` rather
   than collapsing to a misleading 0.

In [22]:
pt_raw = load_raw("THA25")
n_blank = int(blank_mask(pt_raw["VALUE"]).sum())
pt_raw["VALUE"] = coerce_value(pt_raw["VALUE"])
wk = pt_raw["Week"].map(parse_week)
pt_raw["Year"]    = [y for y, _ in wk]
pt_raw["WeekNum"] = [w for _, w in wk]

# 1. drop empty Week-53 placeholders
wk53 = pt_raw[pt_raw["WeekNum"] == 53]
if wk53["VALUE"].notna().sum() == 0:
    pt_raw = pt_raw[pt_raw["WeekNum"] != 53].copy()
    audit("THA25", f"dropped {len(wk53)} empty 'Week 53' placeholder rows.")

# 2. detect rail-gap years
mode_map = {"Dublin Metro Bus": "bus", "Bus, excluding Dublin Metro": "bus", "Rail": "rail"}
pt_raw["mode"] = pt_raw["Mode of Transport"].map(mode_map)
rail_gap_years = [int(y) for y in sorted(pt_raw["Year"].unique())
                  if pt_raw[(pt_raw.Year == y) & (pt_raw["mode"] == "rail")]["VALUE"].notna().sum() == 0]
if rail_gap_years:
    audit("THA25", f"Rail entirely absent for {rail_gap_years} -> bus-only, flagged.")

pt = (pt_raw.dropna(subset=["mode"])
      .groupby(["Year", "mode"], as_index=False)["VALUE"].sum(min_count=1)
      .pivot(index="Year", columns="mode", values="VALUE").reset_index())
pt.columns.name = None
pt = pt.rename(columns={"bus": "bus_journeys", "rail": "rail_journeys"})
pt["pt_weeks_present"] = pt["Year"].map(pt_raw.groupby("Year")["WeekNum"].nunique())
pt["rail_complete"] = ~pt["Year"].isin(rail_gap_years)
audit("THA25", f"{n_blank} blank weekly cells handled; annual bus/rail for "
               f"{pt.Year.min()}-{pt.Year.max()}.")
print("Rail-gap years:", rail_gap_years)
print(pt.to_string(index=False))

Rail-gap years: [2019]
 Year  bus_journeys  rail_journeys  pt_weeks_present  rail_complete
 2019   188557151.0            NaN                53          False
 2020    97524090.0     18781870.0                53           True
 2021    99411734.0     17072961.0                53           True
 2022   167993544.0     34738117.0                53           True
 2023   206476282.0     43918912.0                52           True
 2024   225628060.0     47988050.0                52           True
 2025   226980852.0     44539399.0                52           True


## 6. THA18 — Private-car vehicle population (Car Dependency numerator)

THA18 is split by engine capacity × fuel × county with **no `all` aggregate
rows**, so the national total is the sum across every breakdown. It also carries
`Kilometres Travelled` (millions) and `Average Kilometres Travelled` under the
same columns; we extract each statistic separately to avoid mixing units.

> Note: despite the proposal's label, THA17/THA18 are CSO *vehicle population &
> kilometres* tables, not roadside traffic counts.

In [23]:
cs_raw = load_raw("THA18")
cs_raw["VALUE"] = coerce_value(cs_raw["VALUE"])
cs_raw["Year"]  = cs_raw["Year"].astype(int)
print("Statistics in THA18:", sorted(cs_raw["Statistic Label"].unique()))

car_pop = (cs_raw[cs_raw["Statistic Label"] == "Vehicle Population"]
           .groupby("Year", as_index=False)["VALUE"].sum()
           .rename(columns={"VALUE": "private_cars"}))
car_pop["private_cars"] = car_pop["private_cars"].astype("int64")
car_km = (cs_raw[cs_raw["Statistic Label"] == "Kilometres Travelled"]
          .groupby("Year", as_index=False)["VALUE"].sum()
          .rename(columns={"VALUE": "private_car_km_million"}))
car_stock = car_pop.merge(car_km, on="Year", how="left")
audit("THA18", "private-car population summed across engine x fuel x county; car-km kept separately.")

# --- Fleet stock split: Traditional vs Non-Traditional vehicles ---
# THA18's own Fuel Type dimension has only 3 values: Petrol, Diesel, and
# "Other fuel types". CSO folds ALL electrified vehicles (BEV, PHEV, HEV)
# into that single "Other" bucket at the stock level - unlike TEM12, which
# breaks new REGISTRATIONS down finely (see fuel_group / FUEL_GROUP_MAP).
# So this gives a genuine Traditional vs Non-Traditional split, not the
# full Petrol/Diesel/Hybrid/PHEV/BEV breakdown - that finer detail only
# exists in the registrations-flow data used later in the notebook.

pop_by_fuel = (cs_raw[cs_raw["Statistic Label"] == "Vehicle Population"]
               .groupby(["Year", "Fuel Type"], as_index=False)["VALUE"].sum())

TRADITIONAL_FUELS = {"Petrol", "Diesel"}

traditional_stock = (pop_by_fuel[pop_by_fuel["Fuel Type"].isin(TRADITIONAL_FUELS)]
                      .groupby("Year", as_index=False)["VALUE"].sum()
                      .rename(columns={"VALUE": "traditional_cars"}))

non_traditional_stock = (pop_by_fuel[~pop_by_fuel["Fuel Type"].isin(TRADITIONAL_FUELS)]
                          .groupby("Year", as_index=False)["VALUE"].sum()
                          .rename(columns={"VALUE": "non_traditional_cars"}))

car_stock = car_stock.merge(traditional_stock, on="Year", how="left") \
                      .merge(non_traditional_stock, on="Year", how="left")

# Sanity check: the two categories must reconcile exactly with the existing
# undifferentiated total, or something has gone wrong upstream.
_check = car_stock["private_cars"] - (car_stock["traditional_cars"] + car_stock["non_traditional_cars"])
assert (_check.abs() < 1).all(), "Traditional + Non-Traditional stock does not reconcile with private_cars total"

audit("THA18", "private-car stock additionally split into Traditional "
               "(Petrol+Diesel) vs Non-Traditional (Other fuel types = "
               "BEV+PHEV+HEV combined) - a coarse 2-way split; CSO's own "
               "THA18 table does not break electrified vehicles down "
               "further at the stock level.")

print(car_stock.to_string(index=False))

Statistics in THA18: ['Average Kilometres Travelled', 'Kilometres Travelled', 'Vehicle Population']
 Year  private_cars  private_car_km_million  traditional_cars  non_traditional_cars
 2018       2117468                   35971           2076264                 41204
 2019       2168101                   35450           2108346                 59755
 2020       2211414                   26106           2129184                 82230
 2021       2248920                   30218           2130722                118198
 2022       2266205                   34957           2100267                165938
 2023       2311510                   35126           2092659                218851


## 7. THA17 — Total vehicle-kilometres (Transport Intensity numerator)

Total vehicle-km across all vehicle types, summed over fuel × vehicle type ×
county. The `Kilometres Travelled` statistic is in **millions of km** — the
correct vehicle-km measure for the intensity KPI.

In [24]:
vk_raw = load_raw("THA17")
vk_raw["VALUE"] = coerce_value(vk_raw["VALUE"])
vk_raw["Year"]  = vk_raw["Year"].astype(int)
print("Vehicle types:", sorted(vk_raw["Type of Vehicle"].unique()))

vehicle_km = (vk_raw[vk_raw["Statistic Label"] == "Kilometres Travelled"]
              .groupby("Year", as_index=False)["VALUE"].sum()
              .rename(columns={"VALUE": "total_vehicle_km_million"}))
tot_veh = (vk_raw[vk_raw["Statistic Label"] == "Vehicle Population"]
           .groupby("Year", as_index=False)["VALUE"].sum()
           .rename(columns={"VALUE": "total_vehicles"}))
tot_veh["total_vehicles"] = tot_veh["total_vehicles"].astype("int64")
vehicle_km = vehicle_km.merge(tot_veh, on="Year", how="left")
audit("THA17", "total vehicle-km & fleet summed across fuel x type x county; km in millions.")
print(vehicle_km.to_string(index=False))

Vehicle types: ['Exempt vehicles', 'Goods vehicles', 'Large PSVs', 'Motor cycles', 'Other vehicles', 'Private cars', 'Small PSVs', 'Tractors & Machinery']
 Year  total_vehicle_km_million  total_vehicles
 2018                     47531         2729001
 2019                     47062         2790775
 2020                     36219         2848026
 2021                     41856         2901404
 2022                     47587         2927539
 2023                     47281         2983207


## 8. TEM12 — Fuel transition of new private cars (2015–2026)

The petrol→EV story. We keep `New Private Cars`, map raw fuel labels to tidy
groups, treat blank fuel cells as 0 cars, and compute BEV+PHEV and electrified
shares. We reconcile summed detail against the table's own `All fuel types`
total, and flag partial years (2026).

In [25]:
fm_raw = load_raw("TEM12")
fm_raw["VALUE"] = coerce_value(fm_raw["VALUE"])
fm_raw = fm_raw[fm_raw["Type of Vehicle Registration"] == "New Private Cars"].copy()

detail = fm_raw[fm_raw["Type of Fuel"] != "All fuel types"].copy()
detail["fuel_group"] = detail["Type of Fuel"].map(FUEL_GROUP_MAP).fillna("Other")
detail["VALUE"] = detail["VALUE"].fillna(0)
detail = add_month_columns(detail, "Month")

fuel_monthly = (detail.groupby(["Year","MonthNum","Date","fuel_group"], as_index=False)["VALUE"]
                .sum().rename(columns={"VALUE": "new_private_cars"}))

# reconciliation vs reported 'All fuel types'
allfuel = (add_month_columns(fm_raw[fm_raw["Type of Fuel"] == "All fuel types"], "Month")
           .groupby(["Year","MonthNum"], as_index=False)["VALUE"].sum()
           .rename(columns={"VALUE": "all_fuel_reported"}))
chk = (fuel_monthly.groupby(["Year","MonthNum"])["new_private_cars"].sum().reset_index()
       .merge(allfuel, on=["Year","MonthNum"], how="left"))
print("Detail vs 'All fuel types' max monthly discrepancy:",
      int((chk["new_private_cars"] - chk["all_fuel_reported"]).abs().max()), "cars")

annual = fuel_monthly.groupby(["Year","fuel_group"], as_index=False)["new_private_cars"].sum()
fuel_annual = annual.pivot(index="Year", columns="fuel_group", values="new_private_cars").fillna(0)
fuel_annual["total"] = fuel_annual.sum(axis=1)
fuel_annual["ev_phev_share"] = fuel_annual.reindex(columns=list(PLUGIN_GROUPS), fill_value=0).sum(axis=1) / fuel_annual["total"]
fuel_annual["electrified_share"] = fuel_annual.reindex(columns=list(ELECTRIFIED_GROUPS), fill_value=0).sum(axis=1) / fuel_annual["total"]
fuel_annual = fuel_annual.reset_index(); fuel_annual.columns.name = None
mpy = detail.groupby("Year")["MonthNum"].nunique()
fuel_annual["months_reported"] = fuel_annual["Year"].map(mpy)
fuel_annual["year_complete"] = fuel_annual["months_reported"] == 12
for yr in fuel_annual.loc[~fuel_annual["year_complete"], "Year"]:
    audit("TEM12", f"year {yr} partial ({int(mpy[yr])}/12 months) — shares valid, totals not YoY-comparable.")
audit("TEM12", "fuel mix of New Private Cars mapped to tidy groups; BEV/PHEV & electrified shares computed.")
print(fuel_annual.round(3).to_string(index=False))

Detail vs 'All fuel types' max monthly discrepancy: 1 cars
 Year  Battery Electric (BEV)  Diesel  Hybrid (HEV)  Other  Petrol  Plug-in Hybrid (PHEV)    total  ev_phev_share  electrified_share  months_reported  year_complete
 2015                   476.0 86103.0        1440.0    6.0 32963.0                  122.0 121110.0          0.005              0.017               12           True
 2016                   392.0 99306.0        2478.0    0.0 39472.0                  283.0 141931.0          0.005              0.022               12           True
 2017                   623.0 82492.0        4280.0    0.0 39391.0                  259.0 127045.0          0.007              0.041               12           True
 2018                  1222.0 65814.0        6630.0    0.0 46776.0                  715.0 121157.0          0.016              0.071               12           True
 2019                  3443.0 53201.0        9579.0    0.0 45761.0                 1321.0 113305.0          0.042   

## 9. TEM22 + TEM23 — Continuous private-car licensing series (2019–2026)

A definitional bridge that is the single biggest cleaning hazard:

* **TEM22 (2019–2021)** has **no aggregate rows** and splits *New* vs *Second-hand*.
  National monthly total = New + Second-hand summed across engine × band × authority.
* **TEM23 (2022–2026)** contains **both detail and `All …` subtotals**. We select
  the national aggregate rows only (`All engine` + `All bands` + `All licensing
  authorities`) — summing everything would multiply-count.

The two are concatenated with explicit `source_table` / `definition` columns so
the 2022 boundary stays transparent.

In [26]:
# TEM22: sum detail (blank = 0 cars)
t22 = load_raw("TEM22")
t22["VALUE"] = coerce_value(t22["VALUE"]).fillna(0)
t22 = add_month_columns(t22, "Month")
m22 = (t22.groupby(["Year","MonthNum","Date"], as_index=False)["VALUE"].sum()
       .rename(columns={"VALUE": "private_cars_licensed"}))
m22["source_table"] = "TEM22"; m22["definition"] = "New + Second-hand (summed detail)"

# TEM23: national aggregate rows only
t23 = load_raw("TEM23")
t23["VALUE"] = coerce_value(t23["VALUE"])
agg = t23[(t23["Engine Capacity cc"] == "All engine capacities with respect to private cars (cc)")
          & (t23["Emission Band"] == "All bands")
          & (t23["Licensing Authority"] == "All licensing authorities")].copy()
agg = add_month_columns(agg, "Month")
m23 = (agg.groupby(["Year","MonthNum","Date"], as_index=False)["VALUE"].sum()
       .rename(columns={"VALUE": "private_cars_licensed"}))
m23["source_table"] = "TEM23"; m23["definition"] = "All private cars (new + imported), national aggregate"

reg_monthly = pd.concat([m22, m23], ignore_index=True).sort_values("Date").reset_index(drop=True)
reg_monthly["private_cars_licensed"] = reg_monthly["private_cars_licensed"].round().astype("int64")
reg_annual = reg_monthly.groupby("Year", as_index=False)["private_cars_licensed"].sum()
rmpy = reg_monthly.groupby("Year")["MonthNum"].nunique()
reg_annual["months_reported"] = reg_annual["Year"].map(rmpy)
reg_annual["year_complete"] = reg_annual["months_reported"] == 12
reg_annual["source_table"] = reg_annual["Year"].map(lambda y: "TEM22" if y <= 2021 else "TEM23")
audit("TEM22/TEM23", "concatenated to one 2019-2026 series; boundary at 2022; TEM23 aggregate rows used to avoid double-counting.")
print(reg_annual.to_string(index=False))

 Year  private_cars_licensed  months_reported  year_complete source_table
 2019                 203674               12           True        TEM22
 2020                 162850               12           True        TEM22
 2021                 169895               12           True        TEM22
 2022                 147916               12           True        TEM23
 2023                 167805               12           True        TEM23
 2024                 178521               12           True        TEM23
 2025                 192401               12           True        TEM23
 2026                  95320                4          False        TEM23


## 10. NaPTAN — Public transport stop layer (optional map)

Cleaning for the dashboard map: keep active stops, drop duplicate `AtcoCode`s,
validate coordinates inside the island-of-Ireland bounding box, and map
`StopType` codes to readable categories.

In [27]:
nap = load_raw("NAPTAN").rename(columns={"": "row_id"})
n0 = len(nap)
for c in ("Latitude", "Longitude"):
    nap[c] = pd.to_numeric(nap[c], errors="coerce")
active = nap[nap["Status"].str.lower() == "active"].copy()
ndup = int(active["AtcoCode"].duplicated().sum())
active = active.drop_duplicates(subset="AtcoCode", keep="first")
in_box = active["Latitude"].between(51.0, 55.6) & active["Longitude"].between(-11.0, -5.0)
n_oob = int((~in_box).sum())
active = active[in_box]
stoptype_map = {"BCT":"Bus stop","BCS":"Bus bay","BCQ":"Bus stand","RLY":"Rail station",
                "PLT":"Rail platform","MET":"Metro","TXR":"Taxi rank","FER":"Ferry","FBT":"Ferry berth"}
active["stop_category"] = active["StopType"].map(stoptype_map).fillna("Other")
keep = ["AtcoCode","CommonName","Street","stop_category","StopType","Latitude","Longitude",
        "Easting","Northing","AdministrativeAreaRef","Status"]
naptan = active[keep].reset_index(drop=True)
audit("NaPTAN", f"{n0} raw -> {len(naptan)} active stops ({ndup} dup AtcoCodes, {n_oob} out-of-bounds removed).")
print("Stop categories:", naptan["stop_category"].value_counts().to_dict())
print(naptan.head().to_string(index=False))

Stop categories: {'Bus stop': 16667, 'Taxi rank': 334, 'Rail station': 152, 'Bus bay': 146, 'Rail platform': 128, 'Ferry': 56, 'Other': 46, 'Metro': 2}
    AtcoCode      CommonName       Street stop_category StopType  Latitude  Longitude Easting Northing AdministrativeAreaRef Status
7010PB003857   Ballymagrorty                   Bus stop      BCT 55.033404  -7.357772  641057   920841                   701 active
7010PB003858        Coshquin                   Bus stop      BCT 55.033575  -7.357816  641054   920860                   701 active
7010PB003859   Culmore Point Culmore Road      Bus stop      BCT 55.045712  -7.274440  646370   922263                   701 active
7010PB003860   Culmore Point Culmore Road      Bus stop      BCT 55.045702  -7.274299  646379   922262                   701 active
 7010B158131 Ulsterbus Depot                   Bus stop      BCT 54.996629  -7.317866  643648   916772                   701 active


In [28]:
# --- Estimated CO2 from fleet composition (Traditional vs Non-Traditional) ---
# Simplified proxy model, not a COPERT-equivalent inventory - appropriate
# scope for this project. Two sourced emission factors:
#   Traditional: 153 g CO2/km - SEAI's current average-car-on-road estimate
#   Non-Traditional: 0.209 kWh/km (Brady & O'Mahony, 2011, Irish EV energy
#     requirement) x 224.1 g CO2/kWh (SEAI, 2026 official grid intensity)
#     = 46.8 g CO2/km well-to-wheel
# LIMITATION: THA18 folds BEV+PHEV+HEV into one "Other fuel types" bucket,
# so this treats the whole non-traditional category as BEV-equivalent.
# Real HEV/PHEV vehicles still burn some fuel, so this is a lower-bound
# estimate for that category - the true figure sits between this and the
# traditional factor.

TRADITIONAL_CO2_FACTOR = 153.0
NON_TRADITIONAL_CO2_FACTOR = 0.209 * 224.1

km_by_fuel = (cs_raw[cs_raw["Statistic Label"] == "Kilometres Travelled"]
              .groupby(["Year", "Fuel Type"], as_index=False)["VALUE"].sum())

traditional_km = (km_by_fuel[km_by_fuel["Fuel Type"].isin(TRADITIONAL_FUELS)]
                  .groupby("Year", as_index=False)["VALUE"].sum()
                  .rename(columns={"VALUE": "traditional_km_million"}))

non_traditional_km = (km_by_fuel[~km_by_fuel["Fuel Type"].isin(TRADITIONAL_FUELS)]
                       .groupby("Year", as_index=False)["VALUE"].sum()
                       .rename(columns={"VALUE": "non_traditional_km_million"}))

co2_estimate = traditional_km.merge(non_traditional_km, on="Year", how="left")
co2_estimate["traditional_co2_tonnes"]     = (co2_estimate["traditional_km_million"] * TRADITIONAL_CO2_FACTOR).round(0)
co2_estimate["non_traditional_co2_tonnes"] = (co2_estimate["non_traditional_km_million"] * NON_TRADITIONAL_CO2_FACTOR).round(0)
co2_estimate["co2_avoided_tonnes"]         = (co2_estimate["non_traditional_km_million"] * (TRADITIONAL_CO2_FACTOR - NON_TRADITIONAL_CO2_FACTOR)).round(0)

car_stock = car_stock.merge(co2_estimate, on="Year", how="left")

print(co2_estimate.to_string(index=False))
audit("THA18", "estimated tailpipe/well-to-wheel CO2 by fleet category using "
               "sourced SEAI emission factors and Brady & O'Mahony (2011)'s "
               "Irish EV energy-requirement estimate - a simplified proxy "
               "model, not a COPERT-equivalent inventory.")

 Year  traditional_km_million  non_traditional_km_million  traditional_co2_tonnes  non_traditional_co2_tonnes  co2_avoided_tonnes
 2018                   35423                         548               5419719.0                     25667.0             58177.0
 2019                   34679                         771               5305887.0                     36111.0             81852.0
 2020                   25333                         773               3875949.0                     36205.0             82064.0
 2021                   28901                        1317               4421853.0                     61684.0            139817.0
 2022                   32760                        2197               5012280.0                    102901.0            233240.0
 2023                   32102                        3024               4911606.0                    141635.0            321037.0


## 11. Integration — annual fact table + KPIs

Join the cleaned annual series on `Year` and compute the three KPIs plus
year-on-year changes. Total PT journeys use a NaN-aware sum so the 2019 rail gap
is reflected in the `pt_total_complete` flag rather than silently understating
the figure.

In [29]:
fact = (population
        .merge(luas[["Year","luas_journeys","luas_complete"]], on="Year", how="left")
        .merge(pt, on="Year", how="left")
        .merge(car_stock, on="Year", how="left")
        .merge(vehicle_km, on="Year", how="left"))

fact["pt_total_journeys"] = fact[["bus_journeys","rail_journeys","luas_journeys"]].sum(axis=1, min_count=1)
fact["pt_total_complete"] = (fact["rail_complete"].astype("boolean")
                             & fact["luas_complete"].astype("boolean"))

fact["car_dependency_index"]      = (fact["private_cars"] / fact["population"] * 1000).round(2)
fact["pt_usage_index"]            = (fact["pt_total_journeys"] / fact["population"]).round(2)
fact["transport_intensity_index"] = (fact["total_vehicle_km_million"] * 1e6 / fact["population"]).round(1)
fact["traditional_cdi"]           = (fact["traditional_cars"] / fact["population"] * 1000).round(2)
fact["non_traditional_cdi"]       = (fact["non_traditional_cars"] / fact["population"] * 1000).round(2)
fact["traditional_co2_tonnes"]     = fact["traditional_co2_tonnes"]
fact["non_traditional_co2_tonnes"] = fact["non_traditional_co2_tonnes"]
fact["co2_avoided_tonnes"]         = fact["co2_avoided_tonnes"]

fact = fact.sort_values("Year").reset_index(drop=True)
for kpi in ("car_dependency_index","pt_usage_index","transport_intensity_index","traditional_cdi","non_traditional_cdi"):
    fact[f"{kpi}_yoy_pct"] = (fact[kpi].pct_change() * 100).round(2)

lo, hi = CORE_WINDOW
fact["in_core_window"] = fact["Year"].between(lo, hi)
fact["period_phase"] = np.select(
    [fact["Year"] <= 2019, fact["Year"].between(2020, 2021)],
    ["Pre-pandemic baseline", "Covid-19 structural break"],
    default="Post-pandemic recovery")
audit("FACT", f"annual fact table with 5 KPIs + YoY; core window {lo}-{hi} tagged.")

show = ["Year","population","private_cars","traditional_cars","non_traditional_cars","pt_total_journeys","total_vehicle_km_million",
        "car_dependency_index","traditional_cdi","non_traditional_cdi","traditional_co2_tonnes","non_traditional_co2_tonnes","co2_avoided_tonnes",
        "pt_usage_index","transport_intensity_index","pt_total_complete","in_core_window","period_phase"]
print(fact[show].to_string(index=False))

 Year  population  private_cars  traditional_cars  non_traditional_cars  pt_total_journeys  total_vehicle_km_million  car_dependency_index  traditional_cdi  non_traditional_cdi  traditional_co2_tonnes  non_traditional_co2_tonnes  co2_avoided_tonnes  pt_usage_index  transport_intensity_index  pt_total_complete  in_core_window              period_phase
 2018     4885000     2117468.0         2076264.0               41204.0         41836966.0                   47531.0                433.46           425.03                 8.43               5419719.0                     25667.0             58177.0            8.56                     9730.0               <NA>           False     Pre-pandemic baseline
 2019     4958300     2168101.0         2108346.0               59755.0        236904382.0                   47062.0                437.27           425.22                12.05               5305887.0                     36111.0             81852.0           47.78                     9491.6   

## 12. Write processed outputs

In [30]:
outputs = {
    "fact_transport_annual.csv": fact,
    "dim_population_annual.csv": population,
    "luas_journeys_annual.csv": luas,
    "public_transport_annual.csv": pt,
    "private_car_stock_annual.csv": car_stock,
    "vehicle_km_annual.csv": vehicle_km,
    "fuel_mix_new_private_cars_monthly.csv": fuel_monthly,
    "fuel_mix_new_private_cars_annual.csv": fuel_annual,
    "private_car_registrations_monthly.csv": reg_monthly,
    "private_car_registrations_annual.csv": reg_annual,
    "naptan_stops_clean.csv": naptan,
}
for name, df in outputs.items():
    df.to_csv(PROCESSED_DIR / name, index=False)
    print(f"wrote {name:45} {len(df):>6,} rows x {df.shape[1]} cols")

wrote fact_transport_annual.csv                          8 rows x 33 cols
wrote dim_population_annual.csv                          8 rows x 2 cols
wrote luas_journeys_annual.csv                           8 rows x 3 cols
wrote public_transport_annual.csv                        7 rows x 5 cols
wrote private_car_stock_annual.csv                       6 rows x 10 cols
wrote vehicle_km_annual.csv                              6 rows x 3 cols
wrote fuel_mix_new_private_cars_monthly.csv            816 rows x 5 cols
wrote fuel_mix_new_private_cars_annual.csv              12 rows x 12 cols
wrote private_car_registrations_monthly.csv             88 rows x 6 cols
wrote private_car_registrations_annual.csv               8 rows x 5 cols
wrote naptan_stops_clean.csv                        17,531 rows x 11 cols


## 13. Governance & data-quality report

Render the audit log and key limitations to `docs/DATA_QUALITY_REPORT.md` — this
feeds the dashboard's Metadata & Governance panel.

In [31]:
from datetime import datetime, timezone
ts = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
limitations = """## Key limitations carried into the dashboard

- **2019 public transport is bus-only.** THA25 added rail later, so 2019 rail is
  missing; `pt_total_complete = False` flags it. Treat 2019 PT as a bus baseline.
- **'Vehicle Population' is registered stock, not live traffic.** Intensity uses
  the `Kilometres Travelled` measure (millions of km).
- **TEM22 -> TEM23 definitional break** at 2022 is bridged but made explicit via
  `source_table` / `definition`.
- **Partial 2026** (TEM12/TEM23 to March) flagged `year_complete = False`.
- **Blank cells**: zero in count tables (TEM12/22), preserved as missing in THA25.
"""
report = (f"# Data Quality & Governance Report\n\n_Generated {ts}._\n\n"
          f"## Audit log\n\n" + "\n".join(AUDIT) + "\n\n" + limitations)
(DOCS_DIR / "DATA_QUALITY_REPORT.md").write_text(report, encoding="utf-8")
print(report)

# Data Quality & Governance Report

_Generated 2026-07-08 23:38 UTC._

## Audit log

- **PEA01** — population from 19 five-year bands x 2 sexes; years 2018-2025.
- **THA25** — Rail entirely absent for [2019] -> bus-only, flagged.
- **THA25** — 61 blank weekly cells handled; annual bus/rail for 2019-2025.
- **THA18** — private-car population summed across engine x fuel x county; car-km kept separately.
- **THA18** — private-car stock additionally split into Traditional (Petrol+Diesel) vs Non-Traditional (Other fuel types = BEV+PHEV+HEV combined) - a coarse 2-way split; CSO's own THA18 table does not break electrified vehicles down further at the stock level.
- **THA17** — total vehicle-km & fleet summed across fuel x type x county; km in millions.
- **TEM12** — year 2026 partial (4/12 months) — shares valid, totals not YoY-comparable.
- **TEM12** — fuel mix of New Private Cars mapped to tidy groups; BEV/PHEV & electrified shares computed.
- **TEM22/TEM23** — concatenated to one 2019-202

---
### Summary

All eight CSO sources plus the NaPTAN spatial layer have been profiled, cleaned,
validated and integrated into `data/processed/`. The headline
`fact_transport_annual.csv` is the single governed table the Power BI / Tableau
dashboard binds to; the monthly fuel and registration series support the Fuel
Transition and registration pages; and `DATA_QUALITY_REPORT.md` documents every
decision for the governance panel.

**Re-run** end-to-end with *Kernel → Restart & Run All* after dropping a fresh
CSO extract into `data/raw/` (or set `TDD_RAW_DIR`).